In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025

## Load dataset

In [2]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [3]:
# keep only 100 participants for rapid prototyping
from spice import SpiceDataset

keep_participants = torch.arange(0, 100)

def keep_subset(dataset, subset):
    participant_ids = dataset.xs[:, 0, 0, -1]
    mask = torch.isin(participant_ids, subset)
    return SpiceDataset(dataset.xs[mask], dataset.ys[mask])

dataset_train = keep_subset(dataset_train, keep_participants)
dataset_test = keep_subset(dataset_test, keep_participants)    

print(f"Shape of dataset: {dataset_train.xs.shape}")
print(f"Number of participants: {dataset_train.n_participants}")
print(f"Number of actions in dataset: {dataset_train.n_actions}")

Shape of dataset: torch.Size([384, 150, 1, 13])
Number of participants: 100
Number of actions in dataset: 4


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025.SpiceModel,
        spice_config=spice_castro2025.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        learning_rate=0.01,
        # l2_rnn=0.001,
        
        sindy_weight=0,
        polynomial_degree=2,
        pruning_frequency=100,
        pruning_ensemble=0.05,
        pruning_threshold=0.01,
        sindy_alpha=0.0001,
        
        device=torch.device('cpu' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 12%|███████▍                                                     | 122/1000 [04:20<31:17,  2.14s/it, L(Train)=16.1032562, L(Val,RNN)=18.4470139, Conv=1.39e+00]

Training interrupted. Continuing with further operations...

Stage 2: SINDy refit


 12%|█▏        | 119/1000 [00:10<01:29,  9.81it/s, loss=393.3478088, n_params=36.99+/-0.10]

In [ ]:
path_spice = 'params/spice_castro2025.pkl'
from spice.precoded import workingmemory as spice_castro2025

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025.SpiceModel,
        spice_config=spice_castro2025.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.001,
        ensemble_size=1,
        
        # sindy_weight=0.1,
        # sindy_alpha=0.0001,
        pruning_frequency=100,
        pruning_threshold=0.01,
        pruning_ensemble=0.05,
        polynomial_degree=2,
        
        verbose=True,
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [10]:
estimator.load_spice(path_spice)
estimator.aggregate_coefficients()

RuntimeError: Error(s) in loading state_dict for SpiceModel:
	Missing key(s) in state_dict: "submodules_rnn.value_reward_chosen._mult_table", "submodules_rnn.value_reward_chosen._linear_indices", "submodules_rnn.value_reward_not_chosen._mult_table", "submodules_rnn.value_reward_not_chosen._linear_indices", "submodules_rnn.value_choice_chosen._mult_table", "submodules_rnn.value_choice_chosen._linear_indices", "submodules_rnn.value_choice_not_chosen._mult_table", "submodules_rnn.value_choice_not_chosen._linear_indices". 
	Unexpected key(s) in state_dict: "sindy_coefficients.value_reward_env", "sindy_coefficients.value_reward_chosen", "sindy_coefficients.value_reward_not_chosen", "sindy_coefficients.value_choice_chosen", "sindy_coefficients.value_choice_not_chosen", "sindy_coefficients.volatility_chosen", "sindy_coefficients.volatility_not_chosen", "submodules_rnn.value_reward_env.weight_n", "submodules_rnn.value_reward_env.bias_n", "submodules_rnn.value_reward_env.damping_coefficient", "

In [ ]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=3)

## Benchmarking

### Castro2025 benchmark model

In [ ]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

In [ ]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

### GRU Model

In [ ]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [ ]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

In [ ]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

# ANALYSIS

In [ ]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_value_trajectories import plot_value_trajectories, plot_value_trajectories_multi
from analysis_generative import analysis_generative_behavior

## Analysis Model Evaluation

In [ ]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

## Analysis Value Trajectories

In [ ]:
fig = plot_value_trajectories(
    dataset=dataset_test,
 
    action_idx=0,
    participant_id=3,
    block_id=2,

    spice_model=estimator,
    benchmark_model=cfs,
    gru_model=gru,

    output_path='results/value_trajectories.png',
)

## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)